# Pipeline de limpieza de Feina Activa

Importamos las librerías necesarias, configuramos las opciones de visualización y cargamos el conjunto de datos crudo de Feina Activa conservando una copia original en memoria para poder auditar los cambios al final del proceso.

In [4]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

ruta_xml = Path("../data/raw/feina_activa/offers_feina_activa_raw.xml")

## Carga de datos

Cargamos el XML crudo de Feina Activa (`offers_feina_activa_raw.xml`) y construimos el `DataFrame` conservando una copia original (`df_feina_raw`) y una copia de trabajo (`df_feina_silver`) para aplicar las transformaciones sin perder la referencia inicial.

In [6]:
tree = ET.parse(ruta_xml)
root = tree.getroot()

registros = []
for rama in root:
    for ad in rama.findall(".//ad"):
        fila = {elem.tag: (elem.text.strip() if elem.text is not None else pd.NA) for elem in ad}
        registros.append(fila)

df_feina_raw = pd.DataFrame(registros)
df_feina_silver = df_feina_raw.copy()

print("Registros cargados:", len(df_feina_raw))
print("Columnas iniciales:", df_feina_raw.shape[1])
display(df_feina_raw.head(3))

Registros cargados: 4890
Columnas iniciales: 20


,id,url,title,content,company,experience,requirements,studies,category,contract,workingHours,date,salaryMin,salaryMax,city,region,disability,postcode,status,origin
0,FA92323338,https://feinaactiva.gencat.cat/search/offers/d...,CAMBRER/A MENJADOR,"Retirar platos de las mesas, servir bebidas y ...",Empresa de restauració,Experiència 0 anys.. Competències / coneixemen...,"anglès (parlat Baix, escrit Baix)",NaN,<NA>,Contracte laboral temporal (2 mesos),Jornada completa,26/08/2026,1416,1550,CALELLA,BARCELONA,No,08370,PUBLISHED,FA
1,FA92323568,https://feinaactiva.gencat.cat/search/offers/d...,ARQUITECTE/ARQUITECTA,Les funcions principals seran:\n • Suport en e...,"Aribau 70, S.L.",Experiència 1 mesos. Experiència prèvia en prà...,"català (parlat Superior, escrit Superior), esp...",NaN,<NA>,Contracte laboral indefinit,Jornada completa,31/08/2026,2000,2416,BARCELONA,BARCELONA,No,08011,PUBLISHED,FA
2,FA92323274,https://feinaactiva.gencat.cat/search/offers/d...,INGENIERO/A TÉCNICO/A O INDUSTRIAL JUNIOR,"Revisión de documentación, desarrollo de inspe...","PROTEJA SISTEMAS DE SEGURIDAD, S.L.",Experiència 1 mesos. No es necesaria experienc...,"Disponibilitat de vehicle, Permisos de conduir: b",llicenciatura o enginyeria - industrial,<NA>,Contracte laboral temporal (12 mesos),Jornada intensiva,26/08/2026,1800,2000,POLINYÀ,BARCELONA,No,08213,PUBLISHED,FA


## Transformación y limpieza

### Eliminación columnas innecesarias

Eliminamos la columna `category` al no contener ningún valor útil en este lote (100 % de ausencias), y prescindimos también de `rama_xml` al ser redundante con origin (FA corresponde íntegramente a feinaactiva y SICAS a sicas).

In [7]:
df_feina_silver = df_feina_silver.drop(columns=["category"])

print("Columnas restantes:", df_feina_silver.shape[1])
print("Lista de columnas:", df_feina_silver.columns.tolist())

Columnas restantes: 19
Lista de columnas: ['id', 'url', 'title', 'content', 'company', 'experience', 'requirements', 'studies', 'contract', 'workingHours', 'date', 'salaryMin', 'salaryMax', 'city', 'region', 'disability', 'postcode', 'status', 'origin']


### Limpieza de espacios y saltos de línea

Normalizamos los campos de texto eliminando espacios en blanco iniciales y finales, y reemplazando saltos de línea (`\r`, `\n`) o espacios múltiples consecutivos por un único espacio para evitar textos fragmentados (especialmente en `content`, `title` y `company`).

In [8]:
columnas_texto = [
    "id", "url", "title", "content", "company",
    "experience", "requirements", "studies", "contract",
    "workingHours", "city", "region", "disability",
    "postcode", "status", "origin"
]

for col in columnas_texto:
    df_feina_silver[col] = (
        df_feina_silver[col]
        .astype("string")
        .str.replace(r"[\r\n]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

print("Saltos de línea restantes en 'content':", int(df_feina_silver["content"].fillna("").str.contains(r"[\r\n]", regex=True).sum()))
display(df_feina_silver[["title", "company", "content", "postcode"]].head(3))

Saltos de línea restantes en 'content': 0


,title,company,content,postcode
0,CAMBRER/A MENJADOR,Empresa de restauració,"Retirar platos de las mesas, servir bebidas y ...",08370
1,ARQUITECTE/ARQUITECTA,"Aribau 70, S.L.",Les funcions principals seran: • Suport en el ...,08011
2,INGENIERO/A TÉCNICO/A O INDUSTRIAL JUNIOR,"PROTEJA SISTEMAS DE SEGURIDAD, S.L.","Revisión de documentación, desarrollo de inspe...",08213


### Conversión de tipos de datos

Convertimos `salaryMin` y `salaryMax` a tipo numérico (`Float64`) para permitir cálculos y filtros salariales conservando los valores nulos, y parseamos date al tipo fecha (`datetime64`) con formato `%d/%m/%Y`. Mantenemos postcode como cadena de texto (`string`) para preservar los ceros iniciales.

In [9]:
for col in ["salaryMin", "salaryMax"]:
    df_feina_silver[col] = pd.to_numeric(df_feina_silver[col], errors="coerce").astype("Float64")

df_feina_silver["date"] = pd.to_datetime(df_feina_silver["date"], format="%d/%m/%Y", errors="coerce")

display(df_feina_silver[["date", "salaryMin", "salaryMax", "postcode"]].dtypes)
display(df_feina_silver[["date", "salaryMin", "salaryMax", "postcode"]].head(5))

date         datetime64[us]
salaryMin           Float64
salaryMax           Float64
postcode             string
dtype: object

,date,salaryMin,salaryMax,postcode
0,2026-08-26,1416.0,1550.0,08370
1,2026-08-31,2000.0,2416.0,08011
2,2026-08-26,1800.0,2000.0,08213
3,2026-09-03,1750.0,2000.0,08402
4,2026-08-26,1515.0,1600.0,08370


### Estandarización de nombres de columna

Convertimos los nombres de las columnas a `snake_case` para mantener una nomenclatura uniforme y cómoda de consultar en la capa Silver y en los pasos posteriores del proyecto.

In [10]:
df_feina_silver.columns = (
    df_feina_silver.columns.str.replace(r"([a-z0-9])([A-Z])", r"\1_\2", regex=True)
    .str.lower()
)

print("Columnas estandarizadas:", df_feina_silver.columns.tolist())

Columnas estandarizadas: ['id', 'url', 'title', 'content', 'company', 'experience', 'requirements', 'studies', 'contract', 'working_hours', 'date', 'salary_min', 'salary_max', 'city', 'region', 'disability', 'postcode', 'status', 'origin']


## Validación

Comprobamos tras la limpieza la integridad del conjunto de datos: número de registros, ausencia de identificadores duplicados, tipos de datos asignados, valores nulos restantes, coherencia en los rangos salariales y rango temporal de las fechas.

In [11]:
col_smin = "salary_min" if "salary_min" in df_feina_silver.columns else "salaryMin"
col_smax = "salary_max" if "salary_max" in df_feina_silver.columns else "salaryMax"

print("Total filas:", len(df_feina_silver))
print("IDs duplicados:", int(df_feina_silver["id"].duplicated().sum()))
print("Filas duplicadas exactas:", int(df_feina_silver.duplicated().sum()))

rangos_invalidos = df_feina_silver[
    df_feina_silver[col_smin].notna()
    & df_feina_silver[col_smax].notna()
    & (df_feina_silver[col_smin] > df_feina_silver[col_smax])
]
print("Rangos salariales invertidos (min > max):", len(rangos_invalidos))

print("Fecha mínima:", df_feina_silver["date"].min())
print("Fecha máxima:", df_feina_silver["date"].max())

print("\nNulos por columna:")
display(df_feina_silver.isna().sum().to_frame("nulos"))

print("\nTipos de datos finales:")
display(df_feina_silver.dtypes.to_frame("dtype"))

Total filas: 4890
IDs duplicados: 0
Filas duplicadas exactas: 0
Rangos salariales invertidos (min > max): 0
Fecha mínima: 2026-08-21 00:00:00
Fecha máxima: 2026-11-19 00:00:00

Nulos por columna:


,nulos
id,0
url,0
title,0
content,0
company,23
experience,366
requirements,1601
studies,4090
contract,0
working_hours,0



Tipos de datos finales:


,dtype
id,string
url,string
title,string
content,string
company,string
experience,string
requirements,string
studies,string
contract,string
working_hours,string


## Salida

Guardamos el conjunto de datos limpio de Feina Activa en la capa Silver (`data/silver/feina_activa/offers_feina_activa_silver.csv`), creando el directorio si aún no existe.

In [12]:
ruta_silver = Path("../data/silver/feina_activa")
ruta_silver.mkdir(parents=True, exist_ok=True)

archivo_salida = ruta_silver / "offers_feina_activa_silver.csv"
df_feina_silver.to_csv(archivo_salida, index=False, encoding="utf-8")

print("Archivo guardado correctamente en:", archivo_salida)

Archivo guardado correctamente en: ..\data\silver\feina_activa\offers_feina_activa_silver.csv


## Resultados y conclusiones

### Comparación de los datos antes y después de la limpieza

Comprobamos la evolución del volumen de registros y columnas tras el proceso de limpieza hacia la capa Silver.

In [14]:
# 1. Resumen global de filas y columnas
resumen_global = pd.DataFrame({
    "feina_activa": {
        "registros_iniciales": len(df_feina_raw),
        "registros_finales": len(df_feina_silver),
        "diferencia_registros": len(df_feina_silver) - len(df_feina_raw),
        "columnas_iniciales": df_feina_raw.shape[1],
        "columnas_finales": df_feina_silver.shape[1],
        "diferencia_columnas": df_feina_silver.shape[1] - df_feina_raw.shape[1],
    }
}).T
display(resumen_global)

# 2. Ejemplo de limpieza de texto (content) y tipos (date, salarios, postcode)
col_smin = "salary_min" if "salary_min" in df_feina_silver.columns else "salaryMin"
col_smax = "salary_max" if "salary_max" in df_feina_silver.columns else "salaryMax"

idx_ejemplo = df_feina_raw["content"].fillna("").str.contains(r"[\r\n]", regex=True)
fila_idx = df_feina_raw[idx_ejemplo].index[0] if idx_ejemplo.any() else 0

print(f"\n--- Ejemplo en fila {fila_idx} (ANTES / RAW) ---")
display(df_feina_raw.loc[[fila_idx], ["id", "date", "salaryMin", "salaryMax", "postcode", "content"]])

print(f"\n--- Ejemplo en fila {fila_idx} (DESPUÉS / SILVER) ---")
display(df_feina_silver.loc[[fila_idx], ["id", "date", col_smin, col_smax, "postcode", "content"]])

# 3. Comparación de nombres de columna
print("\nColumnas antes (raw):", df_feina_raw.columns.tolist())
print("Columnas después (silver):", df_feina_silver.columns.tolist())

,registros_iniciales,registros_finales,diferencia_registros,columnas_iniciales,columnas_finales,diferencia_columnas
feina_activa,4890,4890,0,20,19,-1



--- Ejemplo en fila 1 (ANTES / RAW) ---


,id,date,salaryMin,salaryMax,postcode,content
1,FA92323568,31/08/2026,2000,2416,08011,Les funcions principals seran:\n • Suport en e...



--- Ejemplo en fila 1 (DESPUÉS / SILVER) ---


,id,date,salary_min,salary_max,postcode,content
1,FA92323568,2026-08-31,2000.0,2416.0,08011,Les funcions principals seran: • Suport en el ...



Columnas antes (raw): ['id', 'url', 'title', 'content', 'company', 'experience', 'requirements', 'studies', 'category', 'contract', 'workingHours', 'date', 'salaryMin', 'salaryMax', 'city', 'region', 'disability', 'postcode', 'status', 'origin']
Columnas después (silver): ['id', 'url', 'title', 'content', 'company', 'experience', 'requirements', 'studies', 'contract', 'working_hours', 'date', 'salary_min', 'salary_max', 'city', 'region', 'disability', 'postcode', 'status', 'origin']


### Problemas encontrados y acciones realizadas

| Problema detectado | Acción realizada |
| --- | --- |
| Columna `category` vacía en todo el lote (`100 %` de nulos) | Eliminación de la columna para evitar arrastrar campos sin información útil. |
| Saltos de línea internos y espacios sobrantes en textos (especialmente en `content`) | Normalización de saltos de línea y espacios múltiples, y recorte de espacios extremos con `.str.strip()`. |
| Fechas, salarios y códigos postales sin tipado adecuado | Conversión de `date` a `datetime64` y `salary_min` / `salary_max` a `Float64`, manteniendo `postcode` como `string` para conservar ceros iniciales. |
| Nombres de columnas en formato mixto (*camelCase* y minúsculas) | Estandarización a `snake_case` (`salary_min`, `salary_max`, `working_hours`). |
| Nulos reales en campos como `studies`, `requirements` o salarios, y variantes de texto | Conservación sin imputación artificial ni normalización destructiva en Silver; las reglas específicas quedan para pasos posteriores o capa Gold. |

### Utilidad de los datos para el proyecto

El conjunto limpio de Feina Activa aporta información real y estructurada de ofertas de empleo en la capa Silver (`offers_feina_activa_silver.csv`), con `4.890` registros y `19` columnas estandarizadas.

* `id`, `url`, `title` y `content`: identifican cada oferta, permiten enlazar a su publicación original y recogen el título y la descripción de las funciones del puesto.
* `company`, `experience`, `requirements` y `studies`: aproximan el perfil demandado por la empresa empleadora y las condiciones o formación solicitadas.
* `contract`, `working_hours`, `date`, `salary_min` y `salary_max`: describen el tipo de contrato, la jornada, la fecha de la oferta y el rango salarial disponible en formato tipado (`datetime64`, `Float64` y `string`).
* `city`, `region`, `disability` y `postcode`: permiten ubicar geográficamente las vacantes y consultar las condiciones indicadas sobre discapacidad, conservando los códigos postales como texto con sus ceros iniciales.
* `status` y `origin`: aportan trazabilidad del estado de la oferta y del origen dentro del XML (`FA` o `SICAS`).

Por tanto, estos datos son idóneos para analizar la demanda real de empleo, la distribución territorial, las condiciones contractuales/salariales y cruzar las ofertas con la taxonomía ESCO en fases posteriores del proyecto.

### Información que falta y limitaciones

A pesar de su utilidad, el conjunto de Feina Activa presenta algunas limitaciones importantes en esta capa Silver.

Feina Activa recoge ofertas de empleo concretas, no una taxonomía jerárquica de profesiones ni un catálogo normalizado de competencias. Por tanto, no permite conocer directamente:

* Una codificación estándar o jerarquía unificada de ocupaciones y competencias ESCO sin realizar un proceso posterior de mapeo, clasificación o enriquecimiento.
* La información en los registros donde el campo viene vacío en origen, como ocurre con frecuencia en `studies` (`4.090` nulos), `requirements` (`1.601` nulos), `salary_min` (`1.173` nulos), `salary_max` (`1.160` nulos) o `experience` (`366` nulos).
* La homogeneidad salarial sin contextualizar el puesto, ya que los importes dependen de factores como la jornada (`working_hours`), el tipo de contrato (`contract`) o si se trata de un salario bruto/neto o mensual/anual según el caso.
* El estado o la evolución posterior de cada vacante una vez extraído el XML (por ejemplo, si ya se cubrió o si cambió tras la fecha registrada).

Limitaciones de cobertura y posibles sesgos:

* La cobertura de los campos no es uniforme en todo el *dataset*, por lo que analizar variables con muchos nulos (como `studies` o salarios) sobre el total de `4.890` ofertas puede sesgar las conclusiones si no se tiene en cuenta el volumen de datos informados.
* Los textos libres y campos semi-estructurados (`content`, `requirements`, `studies`, `experience`, `company`) presentan distintas redacciones, formatos y variantes lingüísticas, lo que exige cautela al filtrar o agrupar de forma literal.
* En la capa Silver se ha optado por no imputar nulos artificiales ni aplicar normalizaciones agresivas en textos o nombres para preservar la fidelidad al origen, dejando las reglas de negocio, agrupaciones o enriquecimientos específicos para pasos posteriores o la capa Gold.
* Al corresponder a una extracción concreta del XML de Feina Activa (con fechas entre `2026-08-21` y `2026-11-19`), los resultados reflejan el comportamiento observado en ese lote y en su ámbito geográfico/de plataforma.